# Lesson 19 Lab — Multimodal, Embedding, and Rerank Service Boundaries

**Puzzle:** Should one endpoint expose generation, image inputs, embeddings, and reranking for every model?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

vLLM supports multiple task families, but capability belongs to a model plus configuration—not to the server binary in general. Routing an unsupported task can fail late or produce a contract mismatch.


## 0. Predict before running

1. Classify the local Qwen checkpoint's primary task.
2. Mark Chat, embeddings, rerank, and image routes ready or blocked.
3. Name the dataset required to evaluate each enabled task.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The compatibility probe inspects the local model's architecture and vLLM task/model interfaces, maps requested endpoints to required capabilities, and records that no multimodal or pooling benchmark was run with this text-generation checkpoint.

- Server capability is the intersection of engine and model support.
- Pooling quality uses retrieval/ranking metrics rather than generation tokens.
- Remote media inputs expand the network and parser attack surface.


## 2. Derive the mechanism

Generative models return token sequences; pooling models return embeddings or scores; multimodal models add processors and media payloads. Rerank endpoints require a scoring task and input pair schema. Each family changes batching dimensions, memory, security, and evaluation metrics. Capability discovery should gate route registration.

### Mechanism at a glance

```mermaid
flowchart TD
  M["model architecture + task"] --> C{"capability discovery"}
  C --> G["generation routes"]
  C --> E["embedding routes"]
  C --> R["rerank routes"]
  C --> V["multimodal routes"]
  G --> Q["task-specific quality + SLO gate"]
  E --> Q
  R --> Q
  V --> Q
```

### Walk it step by step

1. **Identify the task.** Read model architecture and native task support.
2. **Register only valid routes.** Do not expose endpoints the model cannot execute.
3. **Use task metrics.** Generation, retrieval, ranking, and vision need different evaluations.
4. **Review input security.** Media and remote URLs require additional controls.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 19
LESSON_TITLE = 'Multimodal, Embedding, and Rerank Service Boundaries'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260831
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | register every endpoint because vLLM exposes it |
| Candidate | register only model-capability routes with task-specific gates |
| Held constant | local checkpoint, installed vLLM, no substitute models, and declared endpoint requirements |
| Measurements | architecture, multimodal metadata, pooling indicators, route readiness, and missing test artifacts |
| Evidence | `compatibility-probe` |

**Experiment:** Build a capability matrix from local config and installed interfaces, preserving unsupported routes as explicit blocks.


## 5. Inspect the experiment code

The code does not call an unsupported endpoint merely to manufacture an error. It derives a conservative matrix and makes every missing model/evaluation artifact visible.

Do not execute until the code matches the frozen table.


In [2]:
cfg=model_config(); architecture=str((cfg.get("architectures") or ["unknown"])[0])
mm_keys=[key for key in cfg if any(token in key.lower() for token in ("vision","image","audio"))]
generate=architecture.endswith("ForCausalLM") or "CausalLM" in architecture
routes={"chat":{"ready":bool(generate),"reason":"causal generation architecture"},
        "embeddings":{"ready":False,"reason":"no pooling checkpoint/evaluation"},
        "rerank":{"ready":False,"reason":"no scoring checkpoint/evaluation"},
        "multimodal":{"ready":bool(mm_keys),"reason":"multimodal config" if mm_keys else "text-only config"}}
metrics={"model":{"architecture":architecture,"multimodal_config_keys":mm_keys},"routes":routes,
         "native_non_generation_tests":0,"required_next_models":["embedding","rerank","multimodal"]}
analysis=(f"Architecture {architecture} enables Chat={routes['chat']['ready']} and blocks embeddings/"
          f"rerank/multimodal={routes['embeddings']['ready']}/{routes['rerank']['ready']}/"
          f"{routes['multimodal']['ready']} pending matching native models and evaluations.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Architecture | Qwen2ForCausalLM |
| Text generation ready | yes |
| Embeddings ready | no |
| Rerank ready | no |
| Multimodal ready | no |
| Native non-generation tests | 0 |


## 7. Explain the result

Architecture Qwen2ForCausalLM enables Chat=True and blocks embeddings/rerank/multimodal=False/False/False pending matching native models and evaluations.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The installed package/API/configuration surface was inspected. Availability or lint success is not equivalent to native feature execution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 19, "title": 'Multimodal, Embedding, and Rerank Service Boundaries', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'A vLLM installation is multi-capability; this checkpoint is not. Route registration must follow native model/task evidence.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 19,
  "title": "Multimodal, Embedding, and Rerank Service Boundaries",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260831
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "model": {
      "architecture": "Qwen2ForCausalLM",
      "multimodal_config_keys": []
    },
    "routes": {
      "chat": {
        "ready": true,
        "reason": "causal generation architecture"
      },
      "embeddings": {
        "ready": false,
        "reason": "no pooling checkpoint/evaluation"
      },
      "rerank": {
        "ready": false,
        "reason": "no scoring checkpoint/evaluation"
      },
      "multimodal": {
        "ready": false,
        "reason": "text-only config"
      }
    },
    "native_non_generation_tests": 0,
    "required_next_models": [
  

## 9. Make the bounded decision

> A vLLM installation is multi-capability; this checkpoint is not. Route registration must follow native model/task evidence.

**Acceptance/rollback:** Publish a route only when the selected model natively executes it and passes task-specific quality, latency, and security tests.

**Failure analysis:** Architecture names alone can be ambiguous, and vLLM may infer tasks dynamically. The conservative probe can produce false negatives until a native model initialization confirms support.


## 10. Extend the evidence

Add one pinned embedding, rerank, and multimodal checkpoint, then run retrieval NDCG/recall, pairwise ranking, image validation, and mixed-batch memory tests.

The full boundary and references are in [`README.md`](README.md).
